## Build a semantic search engine with LangChain

we will build a search engine over a PDF document. This will allow us to retrieve passages in the PDF that are similar to an input query. The guide also includes a minimal RAG implementation on top of the search engine.


### Concepts
we will focuses on retrieval of text data. We will cover the following concepts 

- Documents and document loaders
- Text splitters
- Embeddings
- Vector stores and retrievers.

In [ ]:
!pip install langchain-community pypdf

## 1. Documents and document loaders 

Attributes are : - page_content , metadata,id(optional)

`metadata` attribute can capture information about the source of the document, 

In [1]:
# generate sample documents (text containers with metadata)
from langchain_core.documents import Document
docs = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

### Loading documents 

In [2]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "../example-data/nke-10k-2023.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))


107


`PyPDFLoader` loads one    `Document` object per PDF page. For each, we can easily access: 
- The string contect of the page
- Metadata containing the file name and page number. 

In [10]:
print(f"{docs[1].page_content[:200]}\n")
print(docs[1].metadata)

Table of Contents
As of July 12, 2023, the number of shares of the Registrant's Common Stock outstanding were:
Class A 304,897,252 
Class B 1,225,074,356 
1,529,971,608 
DOCUMENTS INCORPORATED BY REFE

{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-07-20T16:22:00-04:00', 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'source': '../example-data/nke-10k-2023.pdf', 'total_pages': 107, 'page': 1, 'page_label': '2'}


### Splitting 


##### Text Splitter (in Langchain)**
Text splitting breaks large documents into small , manageable chunks before embeddings 

#####  Why It's Needed
LLMs and embedding models have token limits. 
Smaller chunks -> better embeddings -> more accurate search results 

##### What it Does 
Takes big texts splits into overlapping pieces 

```
1000-token chunk
+ 200-token overlap (current chunk will have 200 token of previous chunk)
```

Overlap keeps context between chunks.(Overlap = context safety net)

##### When You Use It
After loading documents, before embeddings.
`Loader -> Text Splitter -> Embeddings -> Vector DB `

**Business Impact** : 
Improves answer quality, speeds retrieval, lowers cost.


(`In short Text splitter = cuts long text into AI-friendly chunks so search stays precise.`)

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))
# print(all_splits[0].page_content[:200])

516


## 2. Embeddings 
The idea is to store numeric vectors that are associated with the text. Given a query, we can embed it as a vector of the same dimension and use vector similarity metrics (such as cosine similarity) to identify related text.


In [ ]:
# installing huggingface langchain module for embeddings
!pip install  langchain-huggingface
# !pip install -qu langchain-google-genai

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
# no api key needed ( just local model download )
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [5]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 384

[-0.024540921673178673, -0.11797184497117996, 0.004213620442897081, 0.018862437456846237, 0.002389112487435341, 0.09116827696561813, 0.03516397997736931, 0.012289775535464287, -0.006725173909217119, -0.03351491317152977]


## 3. Vector Stores
storing vector in vector db

In [ ]:
# install chroma db for vector store
!pip install  langchain-chroma

In [ ]:
from langchain_chroma import Chroma

# instantiated vector store , now we can index the documents
vector_store = Chroma(
    collection_name="pdf_docs_collection",
    embedding_function=embeddings,
    persist_directory="../local_chromaDb/",  # Where to save data locally, remove if not necessary
)

In [ ]:
ids = vector_store.add_documents(documents=all_splits)

# Embeddings typically represent text as a “dense” vector such that texts with similar meanings are geometrically close

In [9]:
# Embeddings typically represent text as a “dense” vector such that texts with similar meanings are geometrically close

# return documents based on similarity to a string query 
results = vector_store.similarity_search(
    "How many distribution centers does Nike have in the US?"
)

print(results[0])

page_content='direct to consumer operations sell products through the following number of retail stores in the United States:
U.S. RETAIL STORES NUMBER
NIKE Brand factory stores 213 
NIKE Brand in-line stores (including employee-only stores) 74 
Converse stores (including factory stores) 82 
TOTAL 369 
In the United States, NIKE has eight significant distribution centers. Refer to Item 2. Properties for further information.
2023 FORM 10-K 2' metadata={'keywords': '0000320187-23-000039; ; 10-K', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'creator': 'EDGAR Filing HTML Converter', 'title': '0000320187-23-000039', 'page_label': '5', 'creationdate': '2023-07-20T16:22:00-04:00', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'moddate': '2023-07-20T16:22:08-04:00', 'page': 4, 'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'source': '../example-data/nke-10k-2023.pdf', 'total_pages': 107, 'start_index': 3125}


Async query : 

In [ ]:
# Async query 


# (non-blocking,means it can run concurrently with other operations)
# usefull when you need to perform multiple searches or other async operations without waiting for each one to complet sequentially 
results = await vector_store.asimilarity_search("When was Nike incorporated?")

print(results[0])

page_content='Table of Contents
PART I
ITEM 1. BUSINESS
GENERAL
NIKE, Inc. was incorporated in 1967 under the laws of the State of Oregon. As used in this Annual Report on Form 10-K (this "Annual Report"), the terms "we," "us," "our,"
"NIKE" and the "Company" refer to NIKE, Inc. and its predecessors, subsidiaries and affiliates, collectively, unless the context indicates otherwise.
Our principal business activity is the design, development and worldwide marketing and selling of athletic footwear, apparel, equipment, accessories and services. NIKE is
the largest seller of athletic footwear and apparel in the world. We sell our products through NIKE Direct operations, which are comprised of both NIKE-owned retail stores
and sales through our digital platforms (also referred to as "NIKE Brand Digital"), to retail accounts and to a mix of independent distributors, licensees and sales' metadata={'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creationdate': '2023-07-20T16:22:00-04:00', 

Return scores : 

In [14]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("What was Nike's revenue in 2023?")
results2 = vector_store.similarity_search_with_score("My name is Aakash and day by day in every way I am getting better and better.")
doc, score = results[0]
doc2, score2 = results2[0]
print(f"Similar one's Score: {score}\n")
print(f"Unsimilar one's Score: {score2}\n")
# print(doc)

Similar one's Score: 0.3934860825538635

Unsimilar one's Score: 1.6739723682403564



Return documents based on similarity to an embedded query:

In [ ]:
# in tihs we are directly providing the embedding vector instead of a text query
embedding = embeddings.embed_query("How were Nike's margins impacted in 2023?")

results = vector_store.similarity_search_by_vector(embedding) 
print(results[0])
